# Process single cell profiles

NOTE: We are normalizing the plates for all samples as we only have three wells associated with the healthy controls, which is insufficient for normalization.

## Import libraries

In [1]:
import pathlib

import pandas as pd

from pycytominer import annotate, normalize, feature_select

## Set paths and variables

In [2]:
# Set the plate to process
plate_id = "CARD-CelIns-CX7_260803130001"

# Directory with QC-labeled profiles
qc_labeled_dir = pathlib.Path("./data/qc_labeled_profiles/").resolve(strict=True)

# Directory to save single-cell profiles
output_dir = pathlib.Path("./data/single_cell_profiles/")
output_dir.mkdir(parents=True, exist_ok=True)

# Path to the platemap for the validation plate
platemap_path = pathlib.Path(
    "../metadata/platemaps/platemap_validation.csv"
).resolve(strict=True)

# Path to the QC-labeled profile for the validation plate
profile_path = (qc_labeled_dir / f"{plate_id}_qc_labeled.parquet").resolve(
    strict=True
)

# operations to perform for feature selection
feature_select_ops = [
    "variance_threshold",
    "correlation_threshold",
    "blocklist",
    "drop_na_columns",
]

## Process data with pycytominer

In [3]:
print("Performing preprocessing on", plate_id)

output_annotated_file = str(output_dir / f"{plate_id}_sc_annotated.parquet")
output_normalized_file = str(output_dir / f"{plate_id}_sc_normalized.parquet")
output_feature_select_file = str(
    output_dir / f"{plate_id}_sc_feature_selected.parquet"
)

profile_df = pd.read_parquet(profile_path)
platemap_df = pd.read_csv(platemap_path)

# Drop all rows in the profiles that failed any Metadata_cqc columns
cqc_columns = [col for col in profile_df.columns if col.startswith("Metadata_cqc")]
if cqc_columns:
    profile_df = profile_df[~profile_df[cqc_columns].any(axis=1)]

print("Performing annotation for", plate_id, "...")
# Step 1: Annotation
annotate(
    profiles=profile_df,
    platemap=platemap_df,
    join_on=["Metadata_well_position", "Image_Metadata_Well"],
    output_file=output_annotated_file,
    output_type="parquet",
)

# Load the annotated parquet file to fix metadata columns names
annotated_df = pd.read_parquet(output_annotated_file)

# Rename columns
annotated_df.rename(columns={"Image_Metadata_Site": "Metadata_Site"}, inplace=True)

# Save back
annotated_df.to_parquet(output_annotated_file, index=False)

# Step 2: Normalization
normalized_df = normalize(
    profiles=output_annotated_file,
    method="standardize",
    output_file=output_normalized_file,
    output_type="parquet",
    samples="all",
)

# Step 3: Feature selection
print("Performing feature selection for", plate_id, "...")
feature_select(
    profiles=normalized_df,
    operation=feature_select_ops,
    na_cutoff=0,
    output_file=output_feature_select_file,
    output_type="parquet",
    blocklist_file="./blocklist_features.txt",
)

print(f"Annotation, normalization, and feature selection complete for {plate_id}")

Performing preprocessing on CARD-CelIns-CX7_260803130001
Performing annotation for CARD-CelIns-CX7_260803130001 ...
Performing feature selection for CARD-CelIns-CX7_260803130001 ...
Annotation, normalization, and feature selection complete for CARD-CelIns-CX7_260803130001


In [4]:
# Check output file
test_df = pd.read_parquet(output_feature_select_file)

print(test_df.shape)
print("Plate:", test_df.Metadata_Plate.unique())
print(
    "Metadata columns:", [col for col in test_df.columns if col.startswith("Metadata_")]
)
test_df.head(2)

(8525, 1015)
Plate: ['CARD-CelIns-CX7_260803130001']
Metadata columns: ['Metadata_well_row', 'Metadata_well_column', 'Metadata_heart_number', 'Metadata_cell_type', 'Metadata_heart_failure_type', 'Metadata_treatment', 'Metadata_Nuclei_Location_Center_X', 'Metadata_Nuclei_Location_Center_Y', 'Metadata_Cells_Location_Center_X', 'Metadata_Cells_Location_Center_Y', 'Metadata_Image_Count_Cells', 'Metadata_ImageNumber', 'Metadata_Plate', 'Metadata_Well', 'Metadata_Cells_Number_Object_Number', 'Metadata_Cytoplasm_Parent_Cells', 'Metadata_Cytoplasm_Parent_Nuclei', 'Metadata_Nuclei_Number_Object_Number', 'Metadata_cqc_failed_oversegmented_nuclei', 'Metadata_cqc_failed_small_cells', 'Metadata_cqc_failed_low_intensity', 'Metadata_cqc_failed_blurry_cells', 'Metadata_cqc_failed_missegmented_cells', 'Metadata_Site']


,Metadata_well_row,Metadata_well_column,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,...,Nuclei_Texture_InverseDifferenceMoment_Golgi_3_02_256,Nuclei_Texture_InverseDifferenceMoment_Mito_3_00_256,Nuclei_Texture_InverseDifferenceMoment_Mito_3_02_256,Nuclei_Texture_SumEntropy_ER_3_01_256,Nuclei_Texture_SumEntropy_Golgi_3_01_256,Nuclei_Texture_SumEntropy_Mito_3_01_256,Nuclei_Texture_SumVariance_DNA_3_01_256,Nuclei_Texture_SumVariance_ER_3_01_256,Nuclei_Texture_SumVariance_Golgi_3_01_256,Nuclei_Texture_SumVariance_Mito_3_01_256
0,B,2,7,nonfailing,nonfailing,DMSO,473.080952,205.374286,479.115377,233.423488,...,0.249943,-0.945947,-0.409678,0.396446,0.117041,0.684549,0.765943,-0.007223,-0.153568,-0.121372
1,B,2,7,nonfailing,nonfailing,DMSO,466.014257,156.873727,443.976272,178.247882,...,-0.448529,0.283541,0.043414,0.095913,0.186356,-0.108226,-0.243320,-0.246649,-0.157003,-0.235028


In [5]:
# Check output file
test_df = pd.read_parquet(output_annotated_file)

print(test_df.shape)
print("Plate:", test_df.Metadata_Plate.unique())
print(
    "Metadata columns:", [col for col in test_df.columns if col.startswith("Metadata_")]
)
test_df.head(2)

(8525, 2497)
Plate: ['CARD-CelIns-CX7_260803130001']
Metadata columns: ['Metadata_well_row', 'Metadata_well_column', 'Metadata_heart_number', 'Metadata_cell_type', 'Metadata_heart_failure_type', 'Metadata_treatment', 'Metadata_Nuclei_Location_Center_X', 'Metadata_Nuclei_Location_Center_Y', 'Metadata_Cells_Location_Center_X', 'Metadata_Cells_Location_Center_Y', 'Metadata_Image_Count_Cells', 'Metadata_ImageNumber', 'Metadata_Plate', 'Metadata_Well', 'Metadata_Cells_Number_Object_Number', 'Metadata_Cytoplasm_Parent_Cells', 'Metadata_Cytoplasm_Parent_Nuclei', 'Metadata_Nuclei_Number_Object_Number', 'Metadata_cqc_failed_oversegmented_nuclei', 'Metadata_cqc_failed_small_cells', 'Metadata_cqc_failed_low_intensity', 'Metadata_cqc_failed_blurry_cells', 'Metadata_cqc_failed_missegmented_cells', 'Metadata_Site']


,Metadata_well_row,Metadata_well_column,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,...,Nuclei_Texture_Variance_ER_3_02_256,Nuclei_Texture_Variance_ER_3_03_256,Nuclei_Texture_Variance_Golgi_3_00_256,Nuclei_Texture_Variance_Golgi_3_01_256,Nuclei_Texture_Variance_Golgi_3_02_256,Nuclei_Texture_Variance_Golgi_3_03_256,Nuclei_Texture_Variance_Mito_3_00_256,Nuclei_Texture_Variance_Mito_3_01_256,Nuclei_Texture_Variance_Mito_3_02_256,Nuclei_Texture_Variance_Mito_3_03_256
0,B,2,7,nonfailing,nonfailing,DMSO,473.080952,205.374286,479.115377,233.423488,...,7.843829,8.309565,1.137339,1.192731,1.111145,1.054388,3.947906,4.072798,4.952354,4.198313
1,B,2,7,nonfailing,nonfailing,DMSO,466.014257,156.873727,443.976272,178.247882,...,4.661686,4.492908,1.280809,1.239836,1.295123,1.300685,1.288376,1.236652,1.424593,1.338906
